# Verdant-Minds Colab Startup Kit

This notebook runs a Drive-persistent cultivation workflow aligned to current scripts and flags.

## Workflow summary
1. Prepare `/content/Verdant-Minds` (clone or fast-forward pull).
2. Install project + Colab helper dependencies.
3. Mount Drive and prepare `/content/drive/MyDrive/Verdant/` persistence paths.
4. Configure providers safely (no plaintext keys in notebook source).
5. Run one or more cultivation loops with resume/fresh logic.
6. Print post-run sanity summary from persisted state.
7. Optionally run emergent scaffolding analysis and generate plots.


## 1) Clone or update repository

What this does:
- clones into `/content/Verdant-Minds` if missing
- otherwise runs `git pull --ff-only`
- adds repo path to `sys.path` for local imports


In [ ]:
from pathlib import Path
import sys
import subprocess

REPO_URL = 'https://github.com/Captainkoopa42/Verdant-Minds.git'
REPO_DIR = Path('/content/Verdant-Minds')

if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    print(f'Updating existing repo at {REPO_DIR}...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a git repository.')
else:
    print(f'Cloning into {REPO_DIR}...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready:', REPO_DIR)

## 2) Install dependencies

What this does:
- installs `requirements.txt`
- installs Colab helper dependencies used by bootstrap + analysis


In [ ]:
from scripts.colab_bootstrap import install_deps
install_deps(repo_dir=REPO_DIR)

## 3) Mount Drive and configure persistence layout

What this does:
- mounts Google Drive at `/content/drive`
- prepares `/content/drive/MyDrive/Verdant/`
- exposes state and outputs paths used by subsequent cells


In [ ]:
from scripts.colab_bootstrap import mount_drive_and_prepare

paths = mount_drive_and_prepare('/content/drive/MyDrive/Verdant')
DRIVE_ROOT = paths['drive_root']
DRIVE_STATE_PATH = paths['state_path']
DRIVE_BASELINE_PATH = paths['baseline_path']
OUTPUTS_ROOT = paths['outputs_root']

print('DRIVE_STATE_PATH =', DRIVE_STATE_PATH)
print('OUTPUTS_ROOT =', OUTPUTS_ROOT)

## 4) Configure provider and run settings (secure key handling)

What this does:
- reads provider keys from environment if already present
- prompts with `getpass` only for missing keys
- sets provider chain and run parameters

Do not hardcode API keys in notebook cells.


In [ ]:
import os
import getpass

# Prefer Colab Secrets or runtime env vars. Prompt only if needed.
for key in ['MISTRAL_API_KEY', 'GROQ_API_KEY', 'ANTHROPIC_API_KEY', 'OPENAI_API_KEY']:
    if not os.environ.get(key):
        value = getpass.getpass(f'{key} (leave blank to skip): ')
        if value:
            os.environ[key] = value

# Provider chain for verdant_llm_cultivator.py
os.environ['VERDANT_PROVIDER_CHAIN'] = os.environ.get('VERDANT_PROVIDER_CHAIN', 'mistral,groq,anthropic,openai,local')

# Run configuration
N_RUNS = int(os.environ.get('VERDANT_N_RUNS', '2'))
CYCLES = int(os.environ.get('VERDANT_CYCLES', '40'))
SEED_TOPIC = os.environ.get('VERDANT_SEED_TOPIC', 'contradiction')
PERTURB_INTERVAL = int(os.environ.get('VERDANT_PERTURB_INTERVAL', '10'))

print('VERDANT_PROVIDER_CHAIN =', os.environ['VERDANT_PROVIDER_CHAIN'])
print({'N_RUNS': N_RUNS, 'CYCLES': CYCLES, 'SEED_TOPIC': SEED_TOPIC, 'PERTURB_INTERVAL': PERTURB_INTERVAL})

## 5) Run cultivator loop(s) with resume/fresh behavior

What this does per run:
- creates `/content/drive/MyDrive/Verdant/outputs/<timestamp>/`
- runs `scripts/verdant_llm_cultivator.py` with `--cycles`, `--seed-topic`, and `--perturbation-interval`
- always sets `--save-state /content/drive/MyDrive/Verdant/verdant_persistent_state.json`
- if state exists: adds `--load-state <state>` (resume)
- if state missing: adds `--initialize-knowledge --fresh`

Interpretation note: resumed cycles continue a persisted trajectory and are not equivalent to a fresh run with the same cycle count.

Code note: `initialize_knowledge` is applied only when `--initialize-knowledge` is set and `--load-state` is absent.


In [ ]:
from scripts.colab_bootstrap import run_cultivator_loop

env_overrides = {
    'VERDANT_PROVIDER_CHAIN': os.environ.get('VERDANT_PROVIDER_CHAIN', 'mistral,groq,anthropic,openai,local'),
}

run_output_dirs = run_cultivator_loop(
    repo_dir='/content/Verdant-Minds',
    state_path=DRIVE_STATE_PATH,
    n_runs=N_RUNS,
    cycles=CYCLES,
    seed_topic=SEED_TOPIC,
    perturb_interval=PERTURB_INTERVAL,
    env_overrides=env_overrides,
)

print('Run output directories:')
for path in run_output_dirs:
    print(' -', path)

## 6) Quick state sanity checks

What this does:
- loads persisted state JSON
- prints memory concept count
- prints top `access_count` concepts
- prints wave-emergent concepts in `creation_time` order


In [ ]:
from scripts.colab_bootstrap import quick_summary
quick_summary(DRIVE_STATE_PATH)

## 7) Optional emergent scaffolding analysis

What this does:
- loads persisted state and identifies emergent concepts
- builds weighted graph and top-k-per-node backbone
- analyzes emergent↔emergent edge temporal orientation (`earlier-share`)
- runs shuffled-time baseline trials
- writes `emergent_scaffolding.png` and `link_age_gaps.png`


In [ ]:
!python /content/Verdant-Minds/scripts/analysis/scaffolding_from_state.py --state "$DRIVE_STATE_PATH" --topk 6 --trials 500

print('Plots saved next to state file by default:')
print(' - emergent_scaffolding.png')
print(' - link_age_gaps.png')